# Preparar Dataset de Incendios para ERA5

Limpia, estandariza y prepara los datos para cruce con variables climáticas.

## 1. Importar librerías

In [21]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

print("✓ Librerías importadas")

✓ Librerías importadas


## 2. Cargar datos

In [22]:
DOWNLOAD_DIR = "./datos_incendios"
OUTPUT_DIR = "./datos_procesados"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

raw_combinado = os.path.join(DOWNLOAD_DIR, "incendios_conaf_raw.csv")

if os.path.exists(raw_combinado):
    print(f"Cargando: {raw_combinado}")
    incendios = pd.read_csv(raw_combinado, low_memory=False)
else:
    # Fallback: combinar aquí mismo desde los CSV por temporada (separador '|')
    season_files = sorted(
        f for f in os.listdir(DOWNLOAD_DIR)
        if f.endswith('.csv') and f[:8].isdigit()
    )
    print(f"'{raw_combinado}' no existe; combinando {len(season_files)} archivos de temporada...")
    incendios = pd.concat(
        (pd.read_csv(os.path.join(DOWNLOAD_DIR, f), sep='|', low_memory=False)
         for f in season_files),
        ignore_index=True
    )

print(f"✓ {incendios.shape[0]} filas × {incendios.shape[1]} columnas")

Cargando: ./datos_incendios\incendios_conaf_raw.csv
✓ 109985 filas × 26 columnas


## 3. Inspeccionar estructura

In [23]:
print("\nColumnas del dataset:")
for idx, col in enumerate(incendios.columns, 1):
    print(f"  {idx:2d}. {col}")


Columnas del dataset:
   1. Región
   2. Provincia
   3. Comuna
   4. Temporada
   5. Nombre
   6. Fecha
   7. Hora inicio
   8. Duración (minutos)
   9. Alerta
  10. Escenario
  11. Causa
  12. Superficie quemada: Pino A [ha]
  13. Superficie quemada: Pino B [ha]
  14. Superficie quemada: Pino C [ha]
  15. Superficie quemada: Eucalípto [ha]
  16. Superficie quemada: Otras plantas [ha]
  17. Superficie quemada: Arbolado [ha]
  18. Superficie quemada: Matorral [ha]
  19. Superficie quemada: Pastizal [ha]
  20. Superficie quemada: Agrícola [ha]
  21. Superficie quemada: Desechos [ha]
  22. Superficie quemada total [ha]
  23. Latitud
  24. Longitud
  25. Datum
  26. archivo_origen


## 4. Detectar y mapear columnas clave

In [24]:
# Mapeo explícito de columnas (los CSV de CONAF tienen nombres estables)
column_mapping = {
    'Fecha': 'fecha_evento',
    'Hora inicio': 'hora_evento',
    'Latitud': 'latitud',
    'Longitud': 'longitud',
    'Superficie quemada total [ha]': 'superficie_ha',
    'Región': 'region',
    'Comuna': 'comuna',
    'Causa': 'causa',
}

# Solo renombrar las columnas que realmente existan
column_mapping = {k: v for k, v in column_mapping.items() if k in incendios.columns}

print("Mapeo aplicado:")
for original, nuevo in column_mapping.items():
    print(f"  '{original}' → '{nuevo}'")

incendios = incendios.rename(columns=column_mapping)

faltan = {'fecha_evento', 'latitud', 'longitud'} - set(incendios.columns)
assert not faltan, f"Faltan columnas clave tras el mapeo: {faltan}"
print("\n✓ Columnas renombradas")

Mapeo aplicado:
  'Fecha' → 'fecha_evento'
  'Hora inicio' → 'hora_evento'
  'Latitud' → 'latitud'
  'Longitud' → 'longitud'
  'Superficie quemada total [ha]' → 'superficie_ha'
  'Región' → 'region'
  'Comuna' → 'comuna'
  'Causa' → 'causa'

✓ Columnas renombradas


## 5. Procesar fechas

In [25]:
print("Ejemplos de fechas originales:")
print(incendios['fecha_evento'].head())

# Los CSV de CONAF vienen en formato ISO (YYYY-MM-DD)
incendios['fecha_evento'] = pd.to_datetime(
    incendios['fecha_evento'], format='%Y-%m-%d', errors='coerce'
)

n_nat = incendios['fecha_evento'].isna().sum()
if n_nat:
    print(f"\n⚠️  {n_nat} fechas no se pudieron parsear (quedaron como NaT)")

incendios['año'] = incendios['fecha_evento'].dt.year
incendios['mes'] = incendios['fecha_evento'].dt.month
incendios['fecha_era5'] = incendios['fecha_evento'].dt.strftime('%Y-%m-%d')

print(f"\n✓ Fechas convertidas")
print(f"Rango: {incendios['fecha_evento'].min()} a {incendios['fecha_evento'].max()}")

Ejemplos de fechas originales:
0    2002-07-05
1    2002-10-25
2    2002-10-27
3    2002-11-02
4    2002-11-02
Name: fecha_evento, dtype: str

✓ Fechas convertidas
Rango: 2002-07-01 00:00:00 a 2020-06-21 00:00:00


## 6. Validar coordenadas

In [26]:
incendios['latitud'] = pd.to_numeric(incendios['latitud'], errors='coerce')
incendios['longitud'] = pd.to_numeric(incendios['longitud'], errors='coerce')

n_antes = len(incendios)

# 1) Descartar filas sin coordenadas
incendios = incendios.dropna(subset=['latitud', 'longitud'])

# 2) Descartar coordenadas fuera del recuadro de Chile continental
#    (filtro grueso para detectar errores de carga: signos, ceros, typos)
incendios = incendios[
    incendios['latitud'].between(-56, -17) &
    incendios['longitud'].between(-76, -66)
].reset_index(drop=True)

print(f"Descartadas {n_antes - len(incendios)} filas por coordenadas nulas o fuera de Chile")
print(f"Latitud:  {incendios['latitud'].min():.4f} a {incendios['latitud'].max():.4f}")
print(f"Longitud: {incendios['longitud'].min():.4f} a {incendios['longitud'].max():.4f}")
print(f"\n✓ Coordenadas validadas ({len(incendios)} filas)")

Descartadas 209 filas por coordenadas nulas o fuera de Chile
Latitud:  -55.2233 a -18.1878
Longitud: -74.1500 a -67.2892

✓ Coordenadas validadas (109776 filas)


## 7. Filtrar 2010-2020 (Registro 10 años)

In [30]:
if 'año' in incendios.columns:
    print(f"Eventos antes del filtro: {len(incendios)}")
    
    incendios = incendios[
        (incendios['año'] >= 2010) & (incendios['año'] <= 2019)
    ].reset_index(drop=True)
    
    print(f"Eventos después del filtro: {len(incendios)}")
    print(f"\nEventos por año:")
    print(incendios['año'].value_counts().sort_index())
    
    print(f"\n✓ Filtrado a 2010-2020")

Eventos antes del filtro: 60530
Eventos después del filtro: 60530

Eventos por año:
año
2010    4160
2011    5534
2012    4292
2013    6551
2014    6109
2015    7319
2016    7452
2017    4864
2018    6198
2019    8051
Name: count, dtype: int64

✓ Filtrado a 2010-2020


## 8. Guardar dataset procesado

In [31]:
OUTPUT_DIR = "./datos_procesados"

output_file = os.path.join(OUTPUT_DIR, "incendios_conaf_2010_2020.csv")
incendios.to_csv(output_file, index=False)

print(f"\n✓ Dataset guardado:")
print(f"  Archivo: {output_file}")
print(f"  Tamaño: {os.path.getsize(output_file) / (1024*1024):.2f} MB")
print(f"  Filas: {len(incendios)}")
print(f"  Columnas: {incendios.shape[1]}")


✓ Dataset guardado:
  Archivo: ./datos_procesados\incendios_conaf_2010_2020.csv
  Tamaño: 14.33 MB
  Filas: 60530
  Columnas: 29


## 9. Resumen final

In [32]:
print("\n" + "="*80)
print("✓ DATOS LISTOS PARA ERA5")
print("="*80)
print(f"""
Dataset final:
  • {len(incendios)} eventos de incendio
  • Período: {incendios['fecha_evento'].min().date()} a {incendios['fecha_evento'].max().date()}
  • Latitud: {incendios['latitud'].min():.2f}° a {incendios['latitud'].max():.2f}°
  • Longitud: {incendios['longitud'].min():.2f}° a {incendios['longitud'].max():.2f}°

Próximo paso: Cruce con variables ERA5
""")
print("="*80)


✓ DATOS LISTOS PARA ERA5

Dataset final:
  • 60530 eventos de incendio
  • Período: 2010-01-01 a 2019-12-31
  • Latitud: -55.22° a -18.19°
  • Longitud: -74.15° a -67.29°

Próximo paso: Cruce con variables ERA5

